In [ ]:
import itertools
import re

def tokenize_and_get_vars(formula_str):
    """Extracts all unique single-letter variables from a formula string."""
    token_pat = re.compile(
        r'<-?->|->|=>|⇒|⇔|↔|¬|~|∧|∨|\band\b|\bor\b|\bnot\b|[A-Za-z]|&|\||\^|\(|\)',
        re.IGNORECASE
    )
    raw_tokens = token_pat.findall(formula_str)
    vars_set = set()
    for t in raw_tokens:
        # Fixed: Only exclude 'v' (since it represents the OR operator ∨)
        if t.isalpha() and len(t) == 1 and t.lower() != 'v':
            vars_set.add(t.upper())
    return vars_set

def evaluate_formula(formula_str, env):
    """Parses and evaluates a propositional logic string using a recursive descent parser."""
    token_pat = re.compile(
        r'<-?->|->|=>|⇒|⇔|↔|¬|~|∧|∨|\band\b|\bor\b|\bnot\b|[A-Za-z]|&|\||\^|\(|\)',
        re.IGNORECASE
    )
    raw_tokens = token_pat.findall(formula_str)

    normalized = []
    for t in raw_tokens:
        tl = t.lower()
        if tl in ['not', '~', '¬']:
            normalized.append('NOT')
        elif tl in ['and', '∧', '&', '^']:
            normalized.append('AND')
        elif tl in ['or', '∨', '|'] or tl == 'v':
            normalized.append('OR')
        elif t in ['->', '=>', '⇒']:
            normalized.append('IMPLIES')
        elif t in ['<->', '⇔', '↔']:
            normalized.append('IFF')
        elif t in ['(', ')']:
            normalized.append(t)
        elif t.isalpha() and len(t) == 1 and tl != 'v':
            normalized.append(t.upper())

    idx = 0

    def peek():
        return normalized[idx] if idx < len(normalized) else None

    def consume(expected=None):
        nonlocal idx
        t = peek()
        if expected and t != expected:
            return None
        idx += 1
        return t

    # Grammar Rules for Logic Precedence
    def parse_iff():
        val = parse_implies()
        while peek() == 'IFF':
            consume()
            right = parse_implies()
            val = (val == right)
        return val

    def parse_implies():
        val = parse_or()
        while peek() == 'IMPLIES':
            consume()
            right = parse_or()
            val = (not val or right)
        return val

    def parse_or():
        val = parse_and()
        while peek() == 'OR':
            consume()
            right = parse_and()
            val = (val or right)
        return val

    def parse_and():
        val = parse_not()
        while peek() == 'AND':
            consume()
            right = parse_not()
            val = (val and right)
        return val

    def parse_not():
        if peek() == 'NOT':
            consume()
            return not parse_not()
        return parse_atom()

    def parse_atom():
        t = peek()
        if t == '(':
            consume('(')
            val = parse_iff()
            consume(')')
            return val
        elif t in env:
            consume()
            return env[t]
        else:
            if t: consume()
            return False

    try:
        return parse_iff()
    except Exception:
        return False

def main():
    print("~, ¬, not")
    print("∧, &, and")
    print("∨, v, or")
    print(" ->, =>")
    print("<->, <=>\n")

    print("Note: Separate multiple KB sentences with commas (e.g., A or C, B or not C)")
    kb_raw = input("Enter Knowledge Base (KB): ").strip()
    alpha_raw = input("Enter Query (α): ").strip()

    if not kb_raw or not alpha_raw:
        print("Error: KB and Query cannot be empty.")
        return

    kb_sentences = [s.strip() for s in kb_raw.split(',') if s.strip()]

    variables = set()
    for s in kb_sentences:
        variables.update(tokenize_and_get_vars(s))
    variables.update(tokenize_and_get_vars(alpha_raw))
    variables = sorted(list(variables))

    print(f"\nVariables detected: {variables}")
    print(f"Generating truth table for {2**len(variables)} models...\n")

    headers = list(variables) + kb_sentences + ["**KB (TOTAL)**", f"**Query (α: {alpha_raw})**"]
    header_line = " | ".join(f"{h:<14}" for h in headers)
    print(header_line)
    print("-" * len(header_line))

    combinations = list(itertools.product([False, True], repeat=len(variables)))
    entails = True

    for combo in combinations:
        env = dict(zip(variables, combo))
        sentence_vals = [evaluate_formula(s, env) for s in kb_sentences]
        total_kb_val = all(sentence_vals)
        alpha_val = evaluate_formula(alpha_raw, env)

        if total_kb_val and not alpha_val:
            entails = False
            row_marker = "❌ (Counterexample)"
        elif total_kb_val:
            row_marker = "✅ (Model holds)"
        else:
            row_marker = ""

        var_strs = [f"{str(env[v]):<14}" for v in variables]
        sent_strs = [f"{str(val):<14}" for val in sentence_vals]
        kb_str = f"{str(total_kb_val):<14}"
        alpha_str = f"{str(alpha_val):<14}"

        row_line = " | ".join(var_strs + sent_strs + [kb_str, alpha_str])
        print(f"{row_line} {row_marker}")

    print("-" * len(header_line))

    if entails:
        print(f"\n👉 CONCLUSION: KB ⊨ α is TRUE")
    else:
        print(f"\n👉 CONCLUSION: KB ⊨ α is FALSE")

if __name__ == "__main__":
    main()


   TRUTH-TABLE ENUMERATION INFERENCE SOLVER (TT-ENTAILS)
Supported Operators:
  • Negation:    ~, ¬, not
  • Conjunction: ∧, &, and
  • Disjunction: ∨, v, or
  • Implication: ->, =>
  • Biconditional: <->, <=>

Note: Separate multiple KB sentences with commas (e.g., A or C, B or not C)
Enter Knowledge Base (KB): A or C,B or not C
Enter Query (α): A or B

Variables detected: ['A', 'B', 'C']
Generating truth table for 8 models...

A              | B              | C              | A or C         | B or not C     | **KB (TOTAL)** | **Query (α: A or B)**
---------------------------------------------------------------------------------------------------------------------------
False          | False          | False          | False          | True           | False          | False          
False          | False          | True           | True           | False          | False          | False          
False          | True           | False          | False          | True           